In [1]:
import numpy as np
from tqdm import tqdm

import pandas as pd
from math import sqrt
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score

import warnings
from sklearn.exceptions import ConvergenceWarning
import optuna

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data = pd.read_csv("cars_end.csv")
data

,manufacturer_name,transmission,odometer_value,year_produced,engine_has_gas,engine_capacity,has_warranty,state,drivetrain,price_usd,...,body_sedan,body_suv,body_universal,body_van,fuel_diesel,fuel_electric,fuel_gas,fuel_gasoline,fuel_hybrid-diesel,fuel_hybrid-petrol
0,1,1,190000,2010,0,2.5,0,1,2,10900.00,...,0,0,1,0,0,0,0,1,0,0
1,1,1,290000,2002,0,3.0,0,1,2,5000.00,...,0,0,1,0,0,0,0,1,0,0
2,1,1,402000,2001,0,2.5,0,1,2,2800.00,...,0,1,0,0,0,0,0,1,0,0
3,1,0,10000,1999,0,3.0,0,1,2,9999.00,...,1,0,0,0,0,0,0,1,0,0
4,1,1,280000,2001,0,2.5,0,1,2,2134.11,...,0,0,1,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38486,2,1,290000,2000,0,3.5,0,1,0,2750.00,...,1,0,0,0,0,0,0,1,0,0
38487,2,0,321000,2004,0,2.2,0,1,0,4800.00,...,0,0,0,0,1,0,0,0,0,0
38488,2,1,777957,2000,0,3.5,0,1,0,4300.00,...,1,0,0,0,0,0,0,1,0,0
38489,2,0,20000,2001,0,2.0,0,1,0,4000.00,...,0,0,0,0,0,0,0,1,0,0


In [3]:
# 2. Целевая переменная и признаки
target = 'price_usd'  # пример
X = data.drop(columns=[target])
y = data[target]

# 3. Разделение
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Проверка
print('Train:', X_train.shape, y_train.shape ,'Test:', X_test.shape)

Train: (30792, 47) (30792,) Test: (7699, 47)


In [4]:
def scores(model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    return {
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred),
        "RMSE": sqrt(mean_squared_error(y_test, y_pred)),
        "MAPE": mean_absolute_percentage_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred)
    }

In [5]:
def create_pipeline(model, degree=2):
    return Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('scaler', StandardScaler()),
        ('model', model)
    ])


In [6]:
def run_search(model, param_grid, search_type="grid", n_iter=20):
    pipe = create_pipeline(model)

    if search_type == "grid":
        search = GridSearchCV(pipe, param_grid, cv=5, scoring='r2', n_jobs=-1)

    elif search_type == "random":
        search = RandomizedSearchCV(
            pipe,
            param_distributions=param_grid,
            n_iter=n_iter,
            cv=5,
            scoring='r2',
            n_jobs=-1,
            random_state=42
        )

    else:
        raise ValueError("search_type must be 'grid' or 'random'")

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=ConvergenceWarning)
        search.fit(X_train, y_train)

    return search.best_estimator_, search.best_params_

In [7]:
def run_linear_model():
    print("\n🔍 Модель: linear")

    best_score = -np.inf
    best_model = None
    best_degree = None

    for degree in tqdm([1,2,3,4], desc="Подбор степени полинома"):
        pipe = create_pipeline(LinearRegression(), degree)
        pipe.fit(X_train, y_train)
        score = r2_score(y_test, pipe.predict(X_test))
        if score > best_score:
            best_score = score
            best_degree = degree
            best_model = pipe

    metrics = scores(best_model, X_test, y_test)

    result = {
        "model": "linear",
        **metrics,
        "best_params": {"poly__degree": best_degree},
        "optuna_params": None
    }

    return result, best_model

In [8]:
def run_all_models(models, param_grids, search_type="grid"):
    results = []

    for name, model in models.items():
        print(f"\n🔍 Обработка модели: {name}")

        param_grid = param_grids.get(name, {})

        best_model, best_params = run_search(
            model,
            param_grid,
            search_type=search_type
        )

        metrics = scores(best_model, X_test, y_test)

        result = {
            "model": name,
            **metrics,
            "best_params": best_params
        }

        results.append(result)

    return pd.DataFrame(results)

In [9]:
models = {
    'lasso': Lasso(),
    'ridge': Ridge(),
    'elastic': ElasticNet()
}
param_grids = {
    'lasso': {
        'poly__degree': [2, 3, 4],
        'model__alpha': [0.01, 0.1, 1, 10]
    },
    'ridge': {
        'poly__degree': [2, 3, 4],
        'model__alpha': [0.01, 0.1, 1, 10]
    },
    'elastic': {
        'poly__degree': [2, 3, 4],
        'model__alpha': [0.01, 0.1, 1, 10],
        'model__l1_ratio': [0.2, 0.5, 0.8]
    }
}

In [ ]:
run_linear_model()


🔍 Модель: linear


Подбор степени полинома:  50%|█████     | 2/4 [00:01<00:01,  1.48it/s]